# 제주 설문 → 13개 선호 축·필수 제약 변환

표준 라이브러리만 사용하는 **교육용 작은 변환기**입니다. 전체 앱, 전체 설문 구현, 일정·시설 검증기는 아닙니다. 외부 API·모델·파일 다운로드·개인정보 저장을 하지 않습니다. 아래 예제는 허구 응답입니다.

지원 범위: Q07 최대 3개 순위 테마, Q06 pace, PREF-C01 혼잡 회피, PREF-N01 새로움, PREF-I01 실내 선호, 선택적인 1인 하루 예산·하루 보행시간 상한·포함 경험입니다. 날짜·교통·식이·시설 접근성·여행 전체 비용·실제 장소 제약 검증은 구현하지 않았습니다.

각 응답은 `status=answered|unknown|unanswered|indifferent`를 보존합니다. **미정·미응답·상관없음은 수치 0이나 0.5가 아닙니다.** scalar 0.5는 명시적인 중간 응답일 때만 사용합니다. Q07 미선택 항목의 0은 싫다는 뜻이 아닙니다. 명시적 부정 선호 변환은 이 예제에서 지원하지 않습니다.


In [1]:
import math
import json
from copy import deepcopy

THEMES = ("coast", "nature", "local_food", "cafe", "culture", "art", "outdoor", "learning", "neighborhood")
SCALARS = ("pace", "crowd_avoidance", "novelty", "indoor_preference")
STATUSES = {"answered", "unknown", "unanswered", "indifferent"}
WEIGHTS = {"themes": .60, "pace": .15, "crowd_avoidance": .10, "novelty": .10, "indoor_preference": .05}
QUESTIONS = {"themes": "Q07", "pace": "Q06", "crowd_avoidance": "PREF-C01", "novelty": "PREF-N01", "indoor_preference": "PREF-I01", "budget": "Q05", "walking": "Q06/F03", "experience": "F09"}

def read_answer(raw):
    if raw is None:
        return {"status": "unanswered"}
    if not isinstance(raw, dict) or raw.get("status") not in STATUSES:
        raise ValueError("응답 status가 필요합니다")
    if raw["status"] != "answered" and raw.get("value") is not None:
        raise ValueError("미정·미응답·상관없음에 수치를 부여하지 마세요")
    return deepcopy(raw)

def finite_number(value):
    return type(value) in (int, float) and math.isfinite(value)

def survey_to_profile(answers, *, survey_version="survey-draft-v1", profile_version=1):
    """지원 입력을 결정적으로 변환한다. 추천·LLM 추론·장소 적합성 판정은 하지 않는다."""
    allowed = set(QUESTIONS) | {"legacy_walk"}
    if not isinstance(answers, dict) or set(answers) - allowed:
        raise ValueError("지원하지 않는 입력 키가 있습니다")
    if not isinstance(survey_version, str) or not survey_version.strip() or type(profile_version) is not int or profile_version < 1:
        raise ValueError("유효한 설문/프로필 버전이 필요합니다")
    if answers.get("legacy_walk") not in (None, "none"):
        raise ValueError("이 예제는 legacy walk:none만 지원합니다")
    out = {"schemaVersion": "educational-profile-v1", "surveyVersion": survey_version,
           "profileVersion": profile_version, "preferences": {"themes": None},
           "responseStates": {}, "evidence": {}, "hardConstraints": {},
           "softRequirements": {}, "unanswered": [], "followups": []}
    for key in QUESTIONS:
        a = read_answer(answers.get(key))
        out["responseStates"][key] = a["status"]
        out["evidence"][key] = {"questionId": QUESTIONS[key], "source": "explicit_user_survey",
                                "status": a["status"], "surveyVersion": survey_version}
        if a["status"] == "unanswered":
            out["unanswered"].append(key)
        if key == "themes":
            if a["status"] == "answered":
                ranked = a.get("value")
                if not isinstance(ranked, list) or not 1 <= len(ranked) <= 3 or any(not isinstance(x, str) or x not in THEMES for x in ranked) or len(set(ranked)) != len(ranked):
                    raise ValueError("테마는 중복 없이 1~3개입니다")
                points = dict(zip(ranked, (3, 2, 1)))
                total = sum(points.values())
                out["preferences"][key] = [points.get(t, 0) / total for t in THEMES]
                out["evidence"][key]["rankedSelections"] = ranked
        elif key in SCALARS:
            value = a.get("value") if a["status"] == "answered" else None
            if a["status"] == "answered" and (not finite_number(value) or not 0 <= value <= 1):
                raise ValueError("scalar는 명시적 0~1 수치여야 합니다")
            out["preferences"][key] = value
        elif a["status"] == "answered":
            if type(a.get("strict")) is not bool or type(a.get("confirmed")) is not bool:
                raise ValueError("조건에는 strict/confirmed를 명시하세요")
            value = a.get("value")
            if key in ("budget", "walking"):
                expected = ("KRW", "per_person_per_day") if key == "budget" else ("minutes", "per_person_per_day")
                if (a.get("unit"), a.get("scope")) != expected:
                    raise ValueError("조건의 단위·범위를 확인하세요")
                if type(value) is not int or value < 0:
                    raise ValueError("상한은 0 이상의 정수여야 합니다")
            elif value not in ("cafe_dessert", "olle", "hallasan"):
                raise ValueError("지원 포함 경험이 아닙니다")
            requirement = {k: a[k] for k in ("value", "unit", "scope", "strict", "confirmed") if k in a}
            if a["strict"] and not a["confirmed"]:
                out["followups"].append({"field": key, "reason": "필수 조건 사용자 확인 대기"})
            else:
                target = "hardConstraints" if a["strict"] else "softRequirements"
                out[target][key] = requirement
    if answers.get("legacy_walk") == "none":
        out["legacyPreferences"] = {"walking": "low", "source": "legacy_walk_none"}
        if "walking" not in out["hardConstraints"]:
            out["followups"].append({"field": "walking", "reason": "기존 '거의 안 걸어요'는 낮은 보행 선호입니다. 수치 상한·필수 여부를 직접 확인하세요."})
    observed = [k for k in WEIGHTS if out["preferences"].get(k) is not None]
    out["evidenceCoverage"] = round(sum(WEIGHTS[k] for k in observed), 10)
    return out


테마 순위의 원점수는 3·2·1이며 선택된 점수 합으로 나눕니다. 13개 축을 같은 가중치로 합산하지 않습니다. 테마 9축 전체가 가중치 0.60인 한 블록이고 나머지는 0.15/0.10/0.10/0.05입니다. `evidenceCoverage`는 관측 블록의 정보 충족률이며 추천 확률·품질 점수가 아닙니다.

`strict=true`이고 사용자가 확인한 조건만 `hardConstraints`로 이동합니다. 확인 대기 조건은 `followups`에 남으므로 **후속 질문을 해결하기 전 완성 추천에 사용하면 안 됩니다.** 이 함수는 실제 장소가 제약을 만족한다고 판정하지 않습니다. 자료가 없는 제약은 후속 검사에서 `unknown`이어야 합니다. 예산은 1인 1일 기준 식사·입장료·현지 이동비의 상한 예시이며 항공·숙박 제외입니다. 공동 비용 배분과 총 일정 합산은 별도입니다.


In [2]:
# 허구 응답: 실제 관광객·장소·시설 데이터가 아닙니다.
example = {
    "themes": {"status": "answered", "value": ["coast", "cafe", "culture"]},
    "pace": {"status": "answered", "value": 0},
    "crowd_avoidance": {"status": "answered", "value": .75},
    "novelty": {"status": "unknown"},
    "indoor_preference": {"status": "indifferent"},
    "budget": {"status": "answered", "value": 50000, "unit": "KRW", "scope": "per_person_per_day", "strict": True, "confirmed": True},
    "experience": {"status": "answered", "value": "cafe_dessert", "strict": False, "confirmed": True},
    "legacy_walk": "none"
}
profile = survey_to_profile(example)
print(json.dumps(profile, ensure_ascii=False, indent=2))


{
  "schemaVersion": "educational-profile-v1",
  "surveyVersion": "survey-draft-v1",
  "profileVersion": 1,
  "preferences": {
    "themes": [
      0.5,
      0.0,
      0.0,
      0.3333333333333333,
      0.16666666666666666,
      0.0,
      0.0,
      0.0,
      0.0
    ],
    "pace": 0,
    "crowd_avoidance": 0.75,
    "novelty": null,
    "indoor_preference": null
  },
  "responseStates": {
    "themes": "answered",
    "pace": "answered",
    "crowd_avoidance": "answered",
    "novelty": "unknown",
    "indoor_preference": "indifferent",
    "budget": "answered",
    "walking": "unanswered",
    "experience": "answered"
  },
  "evidence": {
    "themes": {
      "questionId": "Q07",
      "source": "explicit_user_survey",
      "status": "answered",
      "surveyVersion": "survey-draft-v1",
      "rankedSelections": [
        "coast",
        "cafe",
        "culture"
      ]
    },
    "pace": {
      "questionId": "Q06",
      "source": "explicit_user_survey",
      "status":

## 핵심 검사 6개
이 검사는 지원하는 작은 입력 범위만 다룹니다. 전체 문항·날짜·상충 제약·모든 부정 응답·실제 장소 검증을 완성했다는 의미가 아닙니다. 실내 선호를 날씨 정책에서, 체력을 활동 선호에서 자동 추정하는 로직도 없습니다.


In [3]:
def must_reject(value):
    try:
        survey_to_profile(value)
    except ValueError:
        return
    raise AssertionError("거절해야 할 입력입니다")

# 1. 순위 정규화와 입력 불변성
before = deepcopy(example)
p = survey_to_profile(example)
assert example == before and math.isclose(sum(p["preferences"]["themes"]), 1)
assert p["preferences"]["themes"][0] == .5
print("1 PASS: 3·2·1 정규화, 원본 응답 보존")

# 2. 상태 네 종류와 명시적 0 보존
assert p["preferences"]["pace"] == 0
assert p["responseStates"]["novelty"] == "unknown"
assert p["responseStates"]["walking"] == "unanswered"
assert p["responseStates"]["indoor_preference"] == "indifferent"
print("2 PASS: 미정·미응답·상관없음·명시적 0 구분")

# 3. 중복 순위와 잘못된 scalar 거절
must_reject({"themes": {"status": "answered", "value": ["coast", "coast"]}})
must_reject({"pace": {"status": "answered", "value": True}})
must_reject({"pace": {"status": "answered", "value": float("nan")}})
print("3 PASS: 중복·bool·비유한 수 거절")

# 4. 필수/선호와 단위 분리
assert "budget" in p["hardConstraints"] and "experience" in p["softRequirements"]
bad = deepcopy(example); bad["budget"]["scope"] = "whole_trip"
must_reject(bad)
print("4 PASS: 필수/선호 분리, 잘못된 비용 범위 거절")

# 5. 미확인 필수조건과 legacy none의 후속 질문
pending = deepcopy(example); pending["budget"]["confirmed"] = False
r = survey_to_profile(pending)
assert "budget" not in r["hardConstraints"]
assert {f["field"] for f in r["followups"]} >= {"budget", "walking"}
assert r["legacyPreferences"]["walking"] == "low"
print("5 PASS: 확인 대기·낮은 보행 선호를 임의 상한으로 변환하지 않음")

# 6. 블록 coverage·버전·문항 근거
assert p["evidenceCoverage"] == .85
assert p["evidence"]["themes"]["questionId"] == "Q07"
assert p["profileVersion"] == 1 and p["surveyVersion"] == "survey-draft-v1"
assert survey_to_profile({})["evidenceCoverage"] == 0
print("6 PASS: 관측 블록 coverage와 provenance/version")


1 PASS: 3·2·1 정규화, 원본 응답 보존
2 PASS: 미정·미응답·상관없음·명시적 0 구분
3 PASS: 중복·bool·비유한 수 거절
4 PASS: 필수/선호 분리, 잘못된 비용 범위 거절
5 PASS: 확인 대기·낮은 보행 선호를 임의 상한으로 변환하지 않음
6 PASS: 관측 블록 coverage와 provenance/version


## 앱에 연결하기 전
응답 DTO 검증 → 이와 같은 순수 변환 → 누락·확인 대기·충돌 해결 → 페르소나 매칭 → 실제 후보의 필수 제약 검사 순으로 분리합니다. 누락 상태와 근거를 보존하면서 기존 `scripts/persona_model.py`에는 `preferences`만 전달할 수 있습니다. 미응답 축을 제외한 높은 유사도가 충분한 근거를 뜻하지 않으므로 coverage도 함께 표시합니다.

관련 저장소 문서: `docs/04-questionnaire-and-scoring.md`, `docs/08-persona-clustering.md`, `data/persona-prototypes.json`. 이 노트북은 독립 실행되며 해당 파일을 읽을 필요는 없습니다.
